# 06 — Machine Learning y validación cruzada temporal

**Taller de Análisis de Datos I · TP3 · Versión final para el dashboard · 17/11/2026**

Las dos versiones recibidas contienen el mismo código y texto. Esta versión conserva sus cuatro folds expansivos, Ridge con lags y HistGradientBoosting con lags. Integra B0–B3 recalculados en cada entrenamiento para medir cuánto agregan los modelos complejos frente a referencias simples.

No se presupone que HGB gane ni que una corrección mejore. Las conclusiones se generan con la ejecución actual. No se reutilizan los números escritos en versiones anteriores.

## Ejecución

1. Ejecutar primero `04_preparacion_datos_modelado_dashboard.ipynb` hasta generar una ejecución completa.
2. Guardar este notebook en `TP3/notebooks/` o `TP3/`, reiniciar el kernel y ejecutar todas sus celdas.
3. Dependencias: Python 3.10+, pandas 2+, NumPy, scikit-learn 1.2+, matplotlib, joblib e IPython.
4. Las salidas se guardan en `TP3/resultados_dashboard_06/<id_ejecucion>/`. El manifiesto final indica ejecución completa.

Este notebook no depende de `tp3_config.py` ni de variables de otro kernel. **Requiere el artefacto del 04:** no reconstruye silenciosamente otra fuente desde CSV. Busca la última ejecución completa por fecha UTC del identificador; puede fijarse `ID_EJECUCION_04` en la Celda 1 para repetir una fuente específica. Valida contrato, conteos y hashes antes de cargar. Un artefacto alterado o incompatible detiene el proceso.

El 05 no es un requisito de ejecución: sus baselines se vuelven a ajustar aquí por fold. Las salidas del 05 corresponden a otra ventana y no deben compararse directamente con estas métricas de CV.

**Verificación de esta entrega:** ejecución completa con artefactos del 04 generados a partir de datos sintéticos; cuatro folds y seis modelos, claves compatibles con 05, lags invariantes a etiquetas futuras, baselines sin ajuste con validación, respaldo para estaciones nuevas, lectura de pipelines guardados, reconciliación de CSV/hashes y casos de mejoras negativas o MAE original cero. Se entrega sin outputs; falta ejecutarlo sobre los CSV reales del trabajo.


## Celda 1 — Markdown · Fuente y contrato

El 04 exporta **todo el histórico**; no solo entrenamiento purgado, ni una columna `particion`. La CV usa fechas objetivo anteriores al 18/10. El período 18/10–01/11 no se utiliza para ajustar ni evaluar esta CV; noviembre 02–09 tampoco.

Se valida la lista ordenada de 23 predictores y se excluyen explícitamente del modelo las temperaturas observadas, el objetivo y las marcas de recolección de observaciones. Esas columnas se conservan exclusivamente para calcular etiquetas y auditar disponibilidad.

El artefacto pickle debe proceder de una ejecución propia o de confianza. SHA-256 comprueba integridad, no la autoría del archivo. Si faltan las salidas del 04, ejecutarlo primero.


In [ ]:
# Celda 1 — Code | Carga trazable desde 04
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json, hashlib, platform, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

ID_EJECUCION_04 = None  # None = última completa; o pegar el identificador exacto del 04.
VERSION = '06_ml_final_v1'
SEED = 42
MARGEN_H = 2
EDAD_MAX_H = 72
FUERZA_B3 = 100
CORTE_VAL = pd.Timestamp('2026-10-18', tz='UTC')
INICIO_TEST = pd.Timestamp('2026-11-02', tz='UTC')
EXCLUIDOS = set(pd.date_range('2026-10-22','2026-10-26').strftime('%Y-%m-%d'))
FECHAS_HIST = set(pd.date_range('2026-09-10','2026-11-01').strftime('%Y-%m-%d')) - EXCLUIDOS
CLAVE = ['station_id','target_time_utc','horizon_target_hours']
fecha_inicio_ejecucion = datetime.now(timezone.utc)

FECHAS = ['snapshot_time_utc','forecast_generated_at_utc','collection_time_utc',
          'target_time_utc','observation_time_utc','observation_collection_hour_utc',
          'forecast_start_utc','forecast_end_utc']

NUMERICAS = ['latitude','longitude','horizon_target_hours','temperature_forecast_c',
             'temperature_observed_c','relative_humidity_forecast_pct','dewpoint_forecast_c',
             'wind_speed_forecast_kmh','precip_probability_forecast_pct']

COLUMNAS = ['station_id','state','match_type','wind_direction_forecast'] + FECHAS + NUMERICAS

def exigir(condicion, mensaje):
    if not bool(condicion):
        raise ValueError(mensaje)

def sha256_archivo(ruta):
    h = hashlib.sha256()
    with Path(ruta).open('rb') as f:
        for bloque in iter(lambda: f.read(1024*1024), b''):
            h.update(bloque)
    return h.hexdigest()

def huella_claves(df):
    # Convención independiente del orden de entrada; station_id siempre es texto.
    d = df[CLAVE].sort_values(CLAVE, kind='stable').copy()
    d['target_time_utc'] = d.target_time_utc.dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')
    return hashlib.sha256(d.to_csv(index=False, lineterminator='\n').encode('utf-8')).hexdigest()

def ultima_completa(raiz):
    candidatas = []
    for carpeta in sorted((raiz/'resultados_dashboard_04').glob('*')):
        p = carpeta/'manifest.json'
        if not carpeta.is_dir() or not p.is_file():
            continue
        try:
            m = json.loads(p.read_text(encoding='utf-8'))
        except (ValueError, OSError) as e:
            raise ValueError(f'Manifiesto 04 ilegible: {p}. Revisar antes de continuar.') from e
        exigir(isinstance(m, dict), f'Manifiesto 04 inválido: {p}')
        if m.get('estado') != 'completo':
            continue
        identificador = m.get('id_ejecucion', '')
        exigir(identificador == carpeta.name and re.fullmatch(r'[0-9]{8}T[0-9]{6}Z_[A-Za-z0-9]+', identificador),
               f'Identificador 04 no reconocido: {p}')
        fecha = datetime.strptime(identificador.split('_')[0], '%Y%m%dT%H%M%SZ').replace(tzinfo=timezone.utc)
        candidatas.append((fecha, identificador, carpeta, m))
    return max(candidatas, key=lambda x:(x[0],x[1]))[2:] if candidatas else (None, None)

def archivo_verificado(carpeta, manifest, nombre):
    entradas = [a for a in manifest.get('archivos', []) if a.get('nombre') == nombre]
    exigir(len(entradas) == 1, f'{nombre}: falta entrada única en manifiesto 04')
    ruta, registro = carpeta/nombre, entradas[0]
    exigir(ruta.is_file(), f'Falta artefacto 04: {ruta}')
    exigir(ruta.stat().st_size == registro.get('bytes'), f'Tamaño alterado: {ruta}')
    exigir(sha256_archivo(ruta) == registro.get('sha256'), f'SHA-256 incorrecto: {ruta}')
    return ruta

def normalizar_y_controlar(df):
    exigir(set(COLUMNAS) <= set(df.columns), f'Faltan columnas: {sorted(set(COLUMNAS)-set(df.columns))}')
    r = df.copy()
    r['station_id'] = r.station_id.astype('string')
    for c in NUMERICAS:
        r[c] = pd.to_numeric(r[c], errors='coerce')
    for c in FECHAS:
        r[c] = pd.to_datetime(r[c], utc=True, errors='coerce')
    exigir(r[FECHAS].notna().all().all(), 'Marcas temporales nulas/inválidas')
    exigir((r.station_id.notna() & r.station_id.str.strip().ne('')).all(), 'Identificador de estación faltante')
    exigir((r.horizon_target_hours.isin([12,24]) & r.match_type.eq('interval')
            & np.isfinite(r.temperature_forecast_c) & np.isfinite(r.temperature_observed_c)).all(),
           'El histórico contiene filas no elegibles')
    r['horizon_target_hours'] = r.horizon_target_hours.astype('int64')
    fecha = r.target_time_utc.dt.strftime('%Y-%m-%d')
    exigir(fecha.isin(FECHAS_HIST).all(), 'Test, fechas excluidas o fuera del histórico en la entrada')
    if 'archivo_origen' in r:
        exigir(r.archivo_origen.eq('forecast_observation_' + fecha.str.replace('-','_') + '.csv').all(),
               'Fecha objetivo no coincide con archivo de origen')
    exigir(np.isclose((r.target_time_utc-r.snapshot_time_utc).dt.total_seconds()/3600,
                       r.horizon_target_hours, atol=.001, rtol=0).all(), 'Horizonte inconsistente')
    exigir((r.observation_time_utc.ge(r.forecast_start_utc)
            & r.observation_time_utc.lt(r.forecast_end_utc)).all(), 'Observación fuera del intervalo')
    exigir(r.observation_collection_hour_utc.ge(r.observation_time_utc).all(), 'Recolección anterior a observación')
    exigir((r.latitude.between(-90,90) & r.longitude.between(-180,180)).all(), 'Coordenadas inválidas')
    emision = r[['collection_time_utc','forecast_generated_at_utc']].max(axis=1)
    disponible = pd.concat([emision, r.observation_collection_hour_utc+pd.Timedelta(hours=MARGEN_H)],axis=1).max(axis=1)
    exigir(emision.lt(r.target_time_utc).all(), 'Emisión no anterior al objetivo')
    for col, calculado in [('momento_emision_utc',emision), ('error_disponible_utc',disponible)]:
        if col in r:
            exigir(pd.to_datetime(r[col],utc=True,errors='coerce').eq(calculado).all(), f'{col} difiere de la regla 04')
        r[col] = calculado
    error = r.temperature_observed_c-r.temperature_forecast_c
    if 'error_c' in r:
        exigir(np.allclose(pd.to_numeric(r.error_c,errors='coerce'),error,atol=1e-12,rtol=0), 'error_c inconsistente')
    r['error_c'], r['fecha'] = error, fecha
    r[NUMERICAS] = r[NUMERICAS].replace([np.inf,-np.inf],np.nan)
    conocidas = set('N NNE NE ENE E ESE SE SSE S SSW SW WSW W WNW NW NNW'.split())
    direccion = r.wind_direction_forecast.astype('string').str.strip().str.upper()
    exigir(~(r.wind_speed_forecast_kmh.gt(0) & ~direccion.isin(conocidas)).any(), 'Dirección desconocida con viento positivo')
    exigir(not r.duplicated(CLAVE).any(), 'Clave estación/objetivo/horizonte duplicada')
    return r
BASE = ['temperature_forecast_c', 'dewpoint_forecast_c', 'relative_humidity_forecast_pct',
        'wind_speed_forecast_kmh', 'precip_probability_forecast_pct', 'latitude', 'longitude',
        'horizon_target_hours', 'depresion_rocio_c', 'inversion_rocio_flag', 'humedad_cero_flag',
        'viento_calma_flag', 'direccion_viento_sin', 'direccion_viento_cos',
        'hora_utc_sin', 'hora_utc_cos', 'hora_solar_sin', 'hora_solar_cos', 'dia_anio_sin', 'dia_anio_cos']

LAGS = ['lag_error_disponible_c', 'lag_error_edad_h', 'lag_error_faltante_flag']

PREDICTORES = BASE + LAGS

raiz = next((p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents]
             if (p/'resultados_dashboard_04').is_dir()),None)
exigir(raiz is not None, 'No se encuentra resultados_dashboard_04/. Ejecutar 04 desde TP3 antes de continuar.')
if ID_EJECUCION_04 is None:
    carpeta_04, manifest_04 = ultima_completa(raiz)
else:
    exigir(isinstance(ID_EJECUCION_04,str) and re.fullmatch(r'[0-9]{8}T[0-9]{6}Z_[A-Za-z0-9]+',ID_EJECUCION_04),
           'ID_EJECUCION_04 inválido')
    carpeta_04 = raiz/'resultados_dashboard_04'/ID_EJECUCION_04
    exigir((carpeta_04/'manifest.json').is_file(), 'No existe manifiesto para el identificador solicitado')
    manifest_04 = json.loads((carpeta_04/'manifest.json').read_text(encoding='utf-8'))
exigir(manifest_04 is not None and manifest_04.get('estado')=='completo', 'No hay ejecución 04 completa')
exigir(manifest_04.get('id_ejecucion')==carpeta_04.name, 'Identificador 04 inconsistente')
p = manifest_04.get('protocolo',{})
exigir(p.get('version')=='04_dashboard_v1', 'Versión del contrato 04 no soportada')
exigir(p.get('margen_observacion_h')==MARGEN_H and p.get('edad_maxima_lag_h')==EDAD_MAX_H, 'Reglas de lag incompatibles')
exigir(p.get('lags')=='secuencial_eventos_disponibles' and p.get('coincidencias_exactas') is False,
       'Política de lags incompatible')
exigir(p.get('empate_eventos')=='objetivo_mas_reciente'
       and p.get('sin_busqueda_alternativa_si_ultimo_evento_invalido') is True, 'Desempate de lags incompatible')
exigir(pd.Timestamp(p.get('corte_validacion_objetivo_utc'))==CORTE_VAL
       and pd.Timestamp(p.get('inicio_test_objetivo_utc'))==INICIO_TEST
       and set(p.get('excluidos',[]))==EXCLUIDOS, 'Fechas/protocolo distintos del 04')
ruta_features = archivo_verificado(carpeta_04,manifest_04,'features_historicas_tp3.pkl.gz')
ruta_variables = archivo_verificado(carpeta_04,manifest_04,'variables_modelo_lags.json')
variables_ml = json.loads(ruta_variables.read_text(encoding='utf-8'))
exigir(variables_ml==PREDICTORES, 'Lista u orden de predictores distintos del contrato de 23 variables')
fuente = {'id_ejecucion_04':manifest_04['id_ejecucion'],
          'ruta_relativa':str(carpeta_04.relative_to(raiz)),
          'manifest_sha256':sha256_archivo(carpeta_04/'manifest.json'),
          'features_sha256':sha256_archivo(ruta_features),'variables_sha256':sha256_archivo(ruta_variables)}
df_ml = normalizar_y_controlar(pd.read_pickle(ruta_features,compression='gzip')).reset_index(drop=True)
exigir(set(PREDICTORES)<=set(df_ml.columns), 'Faltan predictores exportados')
exigir(len(df_ml)==manifest_04.get('filas',{}).get('historico'), 'Conteo histórico difiere del 04')
exigir(set(df_ml.fecha)==FECHAS_HIST, 'Histórico incompleto')
cobertura = pd.crosstab(df_ml.fecha,df_ml.horizon_target_hours).reindex(columns=[12,24],fill_value=0)
exigir((cobertura>0).all().all(), 'Falta horizonte en una fecha histórica')
exigir(not np.isinf(df_ml[PREDICTORES].to_numpy(float)).any(), 'Features con infinitos')
# Reconstruir las huellas del holdout 04/05: no se entrena ni evalúa ese período.
val_externa = df_ml.loc[df_ml.target_time_utc.ge(CORTE_VAL)]
corte_externo = val_externa.momento_emision_utc.min()
train_externo = df_ml.loc[df_ml.target_time_utc.lt(CORTE_VAL) & df_ml.error_disponible_utc.lt(corte_externo)]
exigir(len(val_externa)==manifest_04['filas']['validacion'] and len(train_externo)==manifest_04['filas']['train'],
       'Particiones externas difieren del 04')
exigir(corte_externo==pd.Timestamp(p['primera_emision_validacion_utc']), 'Corte efectivo externo distinto')
huellas_04_05 = {'train':huella_claves(train_externo),'validacion':huella_claves(val_externa)}
del val_externa,train_externo
# Conservar el orden de eventos exportado para reproducir los desempates del 04.
df_cv = df_ml.loc[df_ml.target_time_utc.lt(CORTE_VAL)].reset_index(drop=True).copy()
exigir(len(df_cv)>0 and df_cv.target_time_utc.max()<CORTE_VAL, 'CV vacía o con fechas externas')
id_ejecucion = fecha_inicio_ejecucion.strftime('%Y%m%dT%H%M%SZ')+'_'+uuid4().hex[:8]
SALIDA = raiz/'resultados_dashboard_06'/id_ejecucion
SALIDA.mkdir(parents=True,exist_ok=False)
print('Fuente 04:',fuente['id_ejecucion_04'])
print('Filas históricas:',len(df_ml),'| candidatas a CV:',len(df_cv))
print('Salidas:',SALIDA)


## Celda 2 — Markdown · Features y lags con la regla del 04

Se reconstruyen las características con las mismas funciones del 04 y se contrastan contra las exportadas. Esto evita aceptar una columna de lag simplemente por su nombre.

El último error debe estar disponible **estrictamente antes** de la emisión, corresponder a la misma estación y horizonte, tener objetivo anterior y edad máxima de 72 h. Los empates se resuelven por objetivo más reciente. Si el último evento disponible no cumple edad u objetivo, el lag queda ausente; no se busca un sustituto.

Los lags simulan un flujo secuencial: una etiqueta de validación o de una fila purgada puede usarse como antecedente de emisiones posteriores una vez disponible. No ajusta los parámetros del modelo. La purga limita el entrenamiento; no elimina información que habría llegado posteriormente al sistema. Los lags se calculan una vez porque dependen de cada emisión, no del fold. Recalcularlos restringiendo eventos al train purgado definiría otro experimento.

La fuente de eventos se limita al período anterior al 18/10. Se compara con los lags del 04 calculados sobre todo el histórico: cualquier diferencia detiene la ejecución. El margen de disponibilidad de 2 h sigue siendo una hipótesis, no el instante real de publicación.


In [ ]:
# Celda 2 — Code | Funciones compartidas con 04 y comprobación del histórico
def construir_features_meteorologicas(df):
    r = df.copy()
    r['depresion_rocio_c'] = (r.temperature_forecast_c-r.dewpoint_forecast_c).clip(lower=0)
    r['inversion_rocio_flag'] = r.dewpoint_forecast_c.gt(r.temperature_forecast_c).astype('int8')
    r['humedad_cero_flag'] = r.relative_humidity_forecast_pct.eq(0).astype('int8')
    r['viento_calma_flag'] = r.wind_speed_forecast_kmh.eq(0).astype('int8')
    mapa = dict(zip('N NNE NE ENE E ESE SE SSE S SSW SW WSW W WNW NW NNW'.split(), np.arange(16)*22.5))
    grados = r.wind_direction_forecast.astype('string').str.strip().str.upper().map(mapa).astype(float)
    exigir(~(r.wind_speed_forecast_kmh.gt(0) & grados.isna()).any(), 'Dirección desconocida con viento positivo')
    angulo = np.deg2rad(grados)
    r['direccion_viento_sin'] = np.sin(angulo).fillna(0).mask(r.viento_calma_flag.eq(1), 0)
    r['direccion_viento_cos'] = np.cos(angulo).fillna(0).mask(r.viento_calma_flag.eq(1), 0)
    hora = r.target_time_utc.dt.hour + r.target_time_utc.dt.minute/60
    solar = (hora + r.longitude/15) % 24
    dias_anio = np.where(r.target_time_utc.dt.is_leap_year, 366, 365)
    for prefijo, fase in [('hora_utc', hora/24), ('hora_solar', solar/24),
                          ('dia_anio', (r.target_time_utc.dt.dayofyear-1)/dias_anio)]:
        r[prefijo+'_sin'] = np.sin(2*np.pi*fase)
        r[prefijo+'_cos'] = np.cos(2*np.pi*fase)
    exigir(not np.isinf(r[BASE].to_numpy(float)).any(), 'Infinitos en features')
    return r

def incorporar_lags(consultas, fuente):
    grupos = ['station_id', 'horizon_target_hours']
    r = consultas.reset_index(drop=True).copy()
    q = r[grupos + ['momento_emision_utc']].copy()
    q['orden'] = np.arange(len(q))
    e = fuente[grupos + ['error_disponible_utc', 'target_time_utc', 'error_c', 'origen_evento']].rename(columns={
        'error_disponible_utc': 'lag_disponible_utc', 'target_time_utc': 'lag_objetivo_utc',
        'error_c': 'lag_error_disponible_c', 'origen_evento': 'lag_origen'})
    e = e.sort_values(['lag_disponible_utc', 'lag_objetivo_utc'], kind='stable')
    lag = pd.merge_asof(q.sort_values('momento_emision_utc'), e,
                        by=grupos, left_on='momento_emision_utc', right_on='lag_disponible_utc',
                        direction='backward', allow_exact_matches=False).sort_values('orden').reset_index(drop=True)
    edad = (lag.momento_emision_utc-lag.lag_objetivo_utc).dt.total_seconds()/3600
    valido = (lag.lag_error_disponible_c.notna() & lag.lag_objetivo_utc.lt(lag.momento_emision_utc)
              & edad.between(0, EDAD_MAX_H))
    exigir(lag.loc[valido, 'lag_disponible_utc'].lt(lag.loc[valido, 'momento_emision_utc']).all(), 'Fuga temporal en lag')
    for col in ['lag_error_disponible_c', 'lag_disponible_utc', 'lag_objetivo_utc', 'lag_origen']:
        r[col] = lag[col].where(valido)
    r['lag_error_edad_h'] = edad.where(valido)
    r['lag_error_faltante_flag'] = (~valido).astype('int8')
    return r

recalculadas = incorporar_lags(construir_features_meteorologicas(df_cv),df_cv)
for col in PREDICTORES:
    exigir(np.allclose(df_cv[col].to_numpy(float),recalculadas[col].to_numpy(float),
                       atol=1e-9,rtol=0,equal_nan=True), f'Feature {col} difiere de la reconstrucción causal del 04')
for col in ['lag_disponible_utc','lag_objetivo_utc']:
    exigir(col in df_cv, f'Falta trazabilidad del lag: {col}')
    original = pd.to_datetime(df_cv[col],utc=True,errors='coerce')
    nuevo = recalculadas[col]
    exigir((original.eq(nuevo) | (original.isna() & nuevo.isna())).all(), f'Trazabilidad {col} difiere')
df_cv = recalculadas
exigir(df_cv.lag_error_faltante_flag.eq(df_cv.lag_error_disponible_c.isna().astype('int8')).all(),
       'Indicador de lag ausente inconsistente')
cobertura_lags = df_cv.groupby(['fecha','horizon_target_hours']).agg(
    registros=('station_id','size'),sin_lag=('lag_error_faltante_flag','sum')).reset_index()
cobertura_lags['cobertura_pct'] = 100*(1-cobertura_lags.sin_lag/cobertura_lags.registros)
display(cobertura_lags.head(12))
print('Features y lags reconciliados con el 04; no se usaron eventos del 18/10 en adelante.')
del recalculadas


## Celda 3 — Markdown · Cuatro folds expansivos y purga

| Fold | Validación: inicio inclusivo | Fin exclusivo |
|---|---|---|
| 1 | 20/09/2026 | 27/09/2026 |
| 2 | 27/09/2026 | 04/10/2026 |
| 3 | 04/10/2026 | 11/10/2026 |
| 4 | 11/10/2026 | 18/10/2026 |

El entrenamiento candidato comienza el 10/09 y crece hasta antes del inicio de cada ventana. Se purgan las etiquetas cuya disponibilidad sea mayor o igual a la primera emisión de esa validación. Los registros de validación no se repiten entre folds; los entrenamientos sí se solapan, como corresponde a una ventana expansiva.

Se exportan filas de entrenamiento, purga y validación por fold, sus huellas y conteos. Los registros anteriores al 20/09 sirven de antecedentes/entrenamiento pero no tienen predicciones fuera de entrenamiento en esta CV.


In [ ]:
# Celda 3 — Code | Construcción de folds
VENTANAS = [('2026-09-20','2026-09-27'),('2026-09-27','2026-10-04'),
            ('2026-10-04','2026-10-11'),('2026-10-11','2026-10-18')]
folds_temporales, resumen_folds, membresias = [], [], []
validaciones_vistas = set()
for numero,(inicio_txt,fin_txt) in enumerate(VENTANAS,1):
    inicio,fin = pd.Timestamp(inicio_txt,tz='UTC'),pd.Timestamp(fin_txt,tz='UTC')
    mascara_val = df_cv.target_time_utc.ge(inicio) & df_cv.target_time_utc.lt(fin)
    idx_val = df_cv.index[mascara_val]
    exigir(len(idx_val)>0, f'Fold {numero}: validación vacía')
    primera_emision = df_cv.loc[idx_val,'momento_emision_utc'].min()
    candidato = df_cv.target_time_utc.lt(inicio)
    seguro = candidato & df_cv.error_disponible_utc.lt(primera_emision)
    idx_train = df_cv.index[seguro]
    exigir(len(idx_train)>0 and len(idx_train.intersection(idx_val))==0, 'Train vacío o solapado con val')
    exigir(not validaciones_vistas.intersection(idx_val), 'Predicciones repetidas entre folds')
    validaciones_vistas.update(idx_val)
    exigir(set(df_cv.loc[idx_train,'horizon_target_hours'])=={12,24}, 'Falta horizonte en train')
    exigir(set(df_cv.loc[idx_val,'horizon_target_hours'])=={12,24}, 'Falta horizonte en val')
    exigir(df_cv.loc[idx_train,'error_disponible_utc'].lt(primera_emision).all(), 'Purga temporal incumplida')
    fechas = set(pd.date_range(inicio,fin-pd.Timedelta(days=1)).strftime('%Y-%m-%d'))
    exigir(set(df_cv.loc[idx_val,'fecha'])==fechas, 'Falta día completo de validación')
    folds_temporales.append({'fold':numero,'train_idx':idx_train.to_numpy(),'val_idx':idx_val.to_numpy(),
                            'primera_emision_val':primera_emision})
    resumen_folds.append({'fold':numero,'inicio_validacion':str(inicio),'fin_exclusivo':str(fin),
                         'train_candidato':int(candidato.sum()),'train_seguro':len(idx_train),
                         'purgados':int((candidato & ~seguro).sum()),'validacion':len(idx_val),
                         'primera_emision_val':str(primera_emision),
                         'train_claves_sha256':huella_claves(df_cv.loc[idx_train]),
                         'val_claves_sha256':huella_claves(df_cv.loc[idx_val])})
    m = df_cv.loc[candidato|mascara_val,CLAVE+['momento_emision_utc','error_disponible_utc']].copy()
    m['fold'] = numero
    m['particion'] = np.where(mascara_val.loc[m.index],'validacion',np.where(seguro.loc[m.index],'train','purgado'))
    membresias.append(m)
df_resumen_cv = pd.DataFrame(resumen_folds)
particiones_folds = pd.concat(membresias,ignore_index=True).sort_values(['fold']+CLAVE,kind='stable')
exigir(len(validaciones_vistas)==int(df_cv.target_time_utc.ge(pd.Timestamp('2026-09-20',tz='UTC')).sum()),
       'Cobertura de predicciones CV incompleta')
display(df_resumen_cv.drop(columns=['train_claves_sha256','val_claves_sha256']))


## Celda 4 — Markdown · Ajuste independiente por fold

Se evalúan B0, B1, B2, B3, Ridge con lags y HGB con lags sobre **exactamente las mismas filas**. Todos los sesgos y preprocesadores se ajustan desde cero con el entrenamiento seguro del fold. B3 usa fuerza 100 y respaldo global para estaciones nuevas.

Se reproduce el preprocesamiento del 04 actual: mediana para las 20 variables base, cero e indicador para lags ausentes; Ridge estandariza los dos grupos (incluido el indicador) y HGB no requiere escalado. Las columnas enteramente faltantes se conservan con `keep_empty_features=True`. Esto corrige la diferencia con las versiones antiguas del 06, que dejaban el indicador sin escalar y convertían todo a float32. Los números anteriores no deben darse por reproducidos.

Ridge: alpha=10, solver=lsqr, max_iter=2000, tol=1e-5. HGB: learning_rate=0.07, max_iter=120, max_leaf_nodes=15, max_depth=6, min_samples_leaf=80, l2_regularization=10, early_stopping=False, random_state=42. No hay búsqueda ni parada aleatoria interna.

Los hiperparámetros ya fueron explorados en etapas previas: esta es una **CV retrospectiva de configuraciones fijas**, no una validación anidada e inédita del proceso de selección. No se modifica el modelo final del 04 ni se utiliza test.


In [ ]:
# Celda 4 — Code | Baselines, Ridge y HGB en los mismos folds
def crear_pipeline(tipo, usar_lags):
    base = Pipeline([('imputar', SimpleImputer(strategy='median', keep_empty_features=True)),
                     ('escalar', StandardScaler() if tipo == 'Ridge' else 'passthrough')])
    transformaciones = [('base', base, BASE)]
    if usar_lags:
        lag = Pipeline([('imputar', SimpleImputer(strategy='constant', fill_value=0, keep_empty_features=True)),
                        ('escalar', StandardScaler() if tipo == 'Ridge' else 'passthrough')])
        transformaciones.append(('lags', lag, LAGS))
    modelo = (Ridge(alpha=10., solver='lsqr', max_iter=2000, tol=1e-5) if tipo == 'Ridge' else
              HistGradientBoostingRegressor(learning_rate=.07, max_iter=120, max_leaf_nodes=15,
                  max_depth=6, min_samples_leaf=80, l2_regularization=10., early_stopping=False, random_state=SEED))
    return Pipeline([('preprocesador', ColumnTransformer(transformaciones)), ('modelo', modelo)])

MODELOS = {'b0':'B0 - Original','b1':'B1 - Sesgo global','b2':'B2 - Sesgo por horizonte',
           'b3':'B3 - Sesgo por estación','ridge_lags':'Ridge con lags','hgb_lags':'HGB con lags'}

def ajustar_baselines(train,val):
    global_c = float(train.error_c.mean())
    por_h = train.groupby('horizon_target_hours').error_c.agg(['mean','count'])
    por_s = train.groupby('station_id').error_c.agg(['mean','count'])
    por_s['correccion_c'] = (por_s['count']*por_s['mean']+FUERZA_B3*global_c)/(por_s['count']+FUERZA_B3)
    respaldo = ~val.station_id.isin(por_s.index)
    pred = {'b0':np.zeros(len(val)),'b1':np.full(len(val),global_c),
            'b2':val.horizon_target_hours.map(por_h['mean']).to_numpy(float),
            'b3':val.station_id.map(por_s.correccion_c).fillna(global_c).to_numpy(float)}
    exigir(all(np.isfinite(a).all() for a in pred.values()), 'Baseline no finito')
    return pred,respaldo,global_c,por_h,por_s

predicciones, parametros_baselines, coeficientes_ridge, auditoria_ajustes = [], [], [], []
parametros_estimadores = {}
for info in folds_temporales:
    numero = info['fold']
    ent = df_cv.loc[info['train_idx']].copy()
    val = df_cv.loc[info['val_idx']].copy()
    pred,respaldo,media,por_h,por_s = ajustar_baselines(ent,val)
    parametros_baselines.append({'fold':numero,'fuerza_B3':FUERZA_B3,'sesgo_global_c':media,
        'horizontes':[{'horizonte_h':int(k),'sesgo_c':float(v['mean']),'n':int(v['count'])} for k,v in por_h.iterrows()],
        'estaciones':[{'station_id':str(k),'media_c':float(v['mean']),'n':int(v['count']),
                       'correccion_B3_c':float(v.correccion_c)} for k,v in por_s.iterrows()]})
    for codigo,tipo in [('ridge_lags','Ridge'),('hgb_lags','HGB')]:
        inicio_fit = time.perf_counter()
        pipeline = crear_pipeline(tipo,usar_lags=True)
        pipeline.fit(ent[PREDICTORES],ent.error_c)
        transformado = pipeline.named_steps['preprocesador'].transform(val[PREDICTORES])
        exigir(transformado.shape==(len(val),len(PREDICTORES)) and np.isfinite(transformado).all(),
               'Preprocesamiento no finito o perdió columnas')
        pred[codigo] = pipeline.predict(val[PREDICTORES])
        exigir(len(pred[codigo])==len(val) and np.isfinite(pred[codigo]).all(), 'Predicción inválida')
        parametros_estimadores[codigo] = pipeline.named_steps['modelo'].get_params()
        archivo = f'modelo_fold_{numero}_{codigo}.joblib'
        joblib.dump(pipeline,SALIDA/archivo)
        auditoria_ajustes.append({'fold':numero,'modelo':codigo,'train':len(ent),'validacion':len(val),
                                  'segundos':time.perf_counter()-inicio_fit,'archivo':archivo,
                                  'ultima_etiqueta_train':str(ent.error_disponible_utc.max()),
                                  'primera_emision_val':str(info['primera_emision_val'])})
        if codigo=='ridge_lags':
            for nombre,coef in zip(pipeline.named_steps['preprocesador'].get_feature_names_out(),pipeline.named_steps['modelo'].coef_):
                coeficientes_ridge.append({'fold':numero,'feature_transformada':nombre,'coeficiente_c':float(coef)})
        del pipeline,transformado
    meta = CLAVE+['state','latitude','longitude','fecha','temperature_forecast_c','temperature_observed_c',
                 'momento_emision_utc','lag_error_disponible_c','lag_error_edad_h','lag_error_faltante_flag',
                 'lag_objetivo_utc','lag_disponible_utc']
    d = val[meta].reset_index(drop=True).copy()
    d['fold'],d['conjunto'] = numero,'cv_temporal'
    d['error_real_c'] = val.error_c.to_numpy(float)
    d['b3_respaldo_global_flag'] = respaldo.to_numpy(dtype='int8')
    for codigo,corr in pred.items():
        d[f'correccion_{codigo}_c'] = corr
        d[f'temperatura_{codigo}_c'] = d.temperature_forecast_c+corr
        d[f'error_residual_{codigo}_c'] = d.error_real_c-corr
    predicciones.append(d)
    print(f'Fold {numero}: train {len(ent):,}, validación {len(val):,}; seis modelos evaluados.')
detalle_cv = pd.concat(predicciones,ignore_index=True).sort_values(CLAVE,kind='stable').reset_index(drop=True)
exigir(not detalle_cv.duplicated(CLAVE).any(), 'Predicciones CV duplicadas')
exigir(len(detalle_cv)==len(validaciones_vistas), 'Faltan predicciones CV')
exigir(huella_claves(detalle_cv)==huella_claves(df_cv.loc[sorted(validaciones_vistas)]), 'Claves de predicción alteradas')


## Celda 5 — Markdown · Métricas comparables y reconciliación

MAE, RMSE y sesgo se calculan por fold, horizonte y sobre todas las predicciones fuera de entrenamiento. La CV consolidada pondera por registros: no promedia RMSE ni porcentajes entre folds. Se informa además un ranking descriptivo por MAE medio de los dos horizontes, como en 04/05.

Las tablas conservan conteos y sumas para reagrupar filtros. Una mejora negativa indica empeoramiento; si el MAE de B0 es cero, el porcentaje queda sin definir. La ventaja frente a B3 es `MAE_B3 − MAE_modelo` en °C: positiva favorece al modelo. No se exige ningún ganador.

Los coeficientes de Ridge se exportan sobre variables transformadas/estandarizadas y dependen del fold. No equivalen a efectos causales ni a coeficientes en las unidades originales. La colinealidad también limita su interpretación.


In [ ]:
# Celda 5 — Code | Métricas aditivas y contraste independiente por fila
SUMAS = ['registros','suma_error_absoluto_c','suma_error_cuadrado_c2','suma_residual_c']
def agregar_metricas(df,dimensiones):
    partes = []
    for codigo in MODELOS:
        r = df[f'error_residual_{codigo}_c']
        d = df[dimensiones].copy()
        d['registros'] = 1
        d['suma_error_absoluto_c'],d['suma_error_cuadrado_c2'],d['suma_residual_c'] = r.abs(),r**2,r
        if dimensiones:
            t = d.groupby(dimensiones,dropna=False,observed=True)[SUMAS].sum().reset_index()
        else:
            t = pd.DataFrame([d[SUMAS].sum()])
        t['registros'] = t.registros.astype('int64')
        t['codigo'],t['modelo'],t['conjunto'] = codigo,MODELOS[codigo],'cv_temporal'
        t['MAE_c'],t['RMSE_c'],t['sesgo_c'] = (t.suma_error_absoluto_c/t.registros,
            np.sqrt(t.suma_error_cuadrado_c2/t.registros),t.suma_residual_c/t.registros)
        partes.append(t)
    m = pd.concat(partes,ignore_index=True)
    for codigo,col in [('b0','MAE_original_c'),('b3','MAE_B3_c')]:
        ref = m.loc[m.codigo.eq(codigo)]
        if dimensiones:
            m = m.merge(ref[dimensiones+['MAE_c']].rename(columns={'MAE_c':col}),on=dimensiones,validate='many_to_one')
        else:
            m[col] = ref.MAE_c.iloc[0]
    m['mejora_MAE_pct'] = 100*(1-m.MAE_c/m.MAE_original_c.where(m.MAE_original_c.ne(0)))
    m['ventaja_vs_B3_c'] = m.MAE_B3_c-m.MAE_c
    return m.sort_values(dimensiones+['codigo'],kind='stable').reset_index(drop=True)

metricas_globales = agregar_metricas(detalle_cv,[])
metricas_horizonte = agregar_metricas(detalle_cv,['horizon_target_hours'])
metricas_fold = agregar_metricas(detalle_cv,['fold'])
metricas_fold_horizonte = agregar_metricas(detalle_cv,['fold','horizon_target_hours'])
metricas_diarias = agregar_metricas(detalle_cv,['fecha','horizon_target_hours'])
metricas_estacion = agregar_metricas(detalle_cv,['station_id','horizon_target_hours'])
metricas_filtros = agregar_metricas(detalle_cv,['fold','fecha','station_id','horizon_target_hours'])

def reconciliar(df):
    exigir(len(df)==len(detalle_cv), 'Cantidad exportada distinta')
    for fold in [None,1,2,3,4]:
        for h in [None,12,24]:
            sel = np.ones(len(df),dtype=bool)
            if fold is not None:sel &= df.fold.eq(fold).to_numpy()
            if h is not None:sel &= df.horizon_target_hours.eq(h).to_numpy()
            ref = (metricas_globales if fold is None and h is None else
                   metricas_horizonte if fold is None else metricas_fold if h is None else metricas_fold_horizonte)
            if fold is not None:ref = ref.loc[ref.fold.eq(fold)]
            if h is not None:ref = ref.loc[ref.horizon_target_hours.eq(h)]
            for codigo in MODELOS:
                r = df.loc[sel,f'error_residual_{codigo}_c'].to_numpy(float)
                exigir(len(r)>0 and np.isfinite(r).all(), 'Residual no finito/vacío')
                esperado = ref.loc[ref.codigo.eq(codigo)]
                exigir(len(esperado)==1 and esperado.iloc[0].registros==len(r), 'Referencia ambigua')
                directo = [np.abs(r).mean(),np.sqrt(np.mean(r**2)),r.mean()]
                exigir(np.allclose(directo,esperado.iloc[0][['MAE_c','RMSE_c','sesgo_c']].to_numpy(float),
                                   atol=1e-9,rtol=0), 'Métricas no reconcilian con las filas')
reconciliar(detalle_cv)
for tabla in [metricas_diarias,metricas_estacion,metricas_filtros,metricas_fold_horizonte]:
    for codigo,g in tabla.groupby('codigo'):
        ref = metricas_globales.set_index('codigo').loc[codigo]
        exigir(g.registros.sum()==ref.registros, 'Conteos agrupados no reconcilian')
        exigir(np.allclose([g.suma_error_absoluto_c.sum()/g.registros.sum(),
                           np.sqrt(g.suma_error_cuadrado_c2.sum()/g.registros.sum()),
                           g.suma_residual_c.sum()/g.registros.sum()],
                           ref[['MAE_c','RMSE_c','sesgo_c']].to_numpy(float),atol=1e-9,rtol=0),
               'Reagrupación no reconcilia')
ranking = metricas_horizonte.groupby(['codigo','modelo'],as_index=False).MAE_c.mean().rename(
    columns={'MAE_c':'MAE_macro_horizontes_c'}).sort_values(['MAE_macro_horizontes_c','codigo'],kind='stable')
display(metricas_globales.round(4))
display(metricas_fold_horizonte.round(4))
display(ranking.round(4))


## Celda 6 — Markdown · Estabilidad temporal

Se muestran mejoras frente a B0 por fold y se cuenta en cuántos de los cuatro folds mejora cada modelo para cada horizonte. Se compara también con B3. Un MAE menor puede coexistir con un sesgo absoluto mayor: las conclusiones deben reflejar ambas métricas.

Los ejes admiten empeoramientos y los porcentajes indefinidos se indican como N/D. Cuatro folds no son cuatro experimentos independientes: comparten entrenamiento y hay dependencia entre observaciones de las mismas estaciones. Estos conteos son descriptivos, no una prueba de significación.


In [ ]:
# Celda 6 — Code | Estabilidad y figuras
estabilidad = metricas_fold_horizonte.assign(
    mejora_frente_B0=lambda d:d.MAE_c.lt(d.MAE_original_c),
    mejora_frente_B3=lambda d:d.MAE_c.lt(d.MAE_B3_c)).groupby(
    ['codigo','modelo','horizon_target_hours'],as_index=False).agg(
        folds_evaluados=('fold','size'),folds_mejora_B0=('mejora_frente_B0','sum'),
        folds_mejora_B3=('mejora_frente_B3','sum'))
display(estabilidad)

def graficar_mejoras(tabla):
    elegidos = ['b3','ridge_lags','hgb_lags']
    fig,ax = plt.subplots(figsize=(11,5))
    for j,codigo in enumerate(elegidos):
        t = tabla.loc[tabla.codigo.eq(codigo)].sort_values('fold')
        x = np.arange(len(t))+(j-1)*.25
        y = t.mejora_MAE_pct.to_numpy(float)
        finitos = np.isfinite(y)
        barras = ax.bar(x[finitos],y[finitos],width=.24,label=MODELOS[codigo])
        ax.bar_label(barras,fmt='%.2f%%',padding=3)
        for pos in x[~finitos]:ax.annotate('N/D',(pos,0),xytext=(0,5),textcoords='offset points',ha='center')
    valores = tabla.loc[tabla.codigo.isin(elegidos),'mejora_MAE_pct'].to_numpy(float)
    valores = valores[np.isfinite(valores)]
    limite_inf = min(0.,float(valores.min())) if len(valores) else 0.
    limite_sup = max(0.,float(valores.max())) if len(valores) else 0.
    margen = max(.6,(limite_sup-limite_inf)*.18)
    ax.set_ylim(limite_inf-margen,limite_sup+margen)
    ax.set_xticks(np.arange(4),[f'Fold {f}' for f in range(1,5)])
    ax.set(title='CV temporal — Reducción del MAE frente al original',ylabel='Reducción del MAE (%)')
    ax.axhline(0,color='gray',linewidth=1)
    ax.grid(axis='y',alpha=.25)
    ax.set_axisbelow(True)
    ax.legend()
    fig.tight_layout()
    return fig
fig = graficar_mejoras(metricas_fold)
fig.savefig(SALIDA/'mejora_por_fold.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
fig,ax = plt.subplots(figsize=(11,5))
metricas_horizonte.pivot(index='modelo',columns='horizon_target_hours',values='MAE_c').plot.bar(ax=ax)
ax.set(title='CV consolidada — MAE por horizonte',ylabel='MAE (°C)',xlabel='Modelo')
ax.tick_params(axis='x',rotation=25)
ax.grid(axis='y',alpha=.25)
fig.tight_layout()
fig.savefig(SALIDA/'mae_consolidado.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)


## Celda 7 — Markdown · Contrato de exportación para el dashboard

| Archivo | Contenido |
|---|---|
| `predicciones_cv_por_fila.csv.gz` | Una fila por estación/objetivo/horizonte con fold, seis predicciones, residuales y trazabilidad del lag |
| `metricas_globales.csv`, `metricas_horizonte.csv` | Tarjetas de CV consolidada |
| `metricas_fold.csv`, `metricas_fold_horizonte.csv` | Rendimiento por ventana temporal |
| `metricas_diarias.csv`, `metricas_estacion.csv`, `metricas_filtros.csv` | Series, mapas y filtros, con numeradores aditivos |
| `resumen_folds.csv`, `particiones_folds.csv.gz` | Cortes, purga y pertenencia de cada fila a cada fold |
| `parametros_baselines_folds.json`, `coeficientes_ridge.csv` | Parámetros aprendidos exclusivamente por fold |
| `modelo_fold_*.joblib` | Pipelines de CV; no reemplazan el modelo final del 04 |
| `diccionario_dashboard.csv`, `manifest.json` | Esquema, protocolo, procedencia, versiones y hashes |

La clave es `station_id`, `target_time_utc`, `horizon_target_hours`. Se exporta ordenada; al cruzar archivos, **unir por clave y verificar conjuntos**, no por número de fila. Leer estaciones como texto y fechas como UTC. El índice de pandas no es una identidad persistente.

Mantener separados `cv_temporal` (06), `validacion` (04/05) y `test_retrospectivo` (04). Comparar la CV de 06 con el holdout del 05 como si fueran las mismas observaciones produciría una conclusión incorrecta. B0–B3 dentro de 06 sí son comparables con Ridge/HGB porque usan los mismos folds.

Para reagrupar: MAE = suma_error_absoluto / registros; RMSE = raíz(suma_error_cuadrado / registros); sesgo = suma_residual / registros. Recalcular mejoras contra B0 en el mismo subconjunto. No sumar registros entre modelos ni promediar porcentajes o RMSE de grupos.


In [ ]:
# Celda 7 — Code | Exportación, lectura de comprobación y diccionario
ruta_detalle = SALIDA/'predicciones_cv_por_fila.csv.gz'
detalle_cv.to_csv(ruta_detalle,index=False,compression='gzip',encoding='utf-8')
verificado = pd.read_csv(ruta_detalle,dtype={'station_id':'string'})
verificado['target_time_utc'] = pd.to_datetime(verificado.target_time_utc,utc=True)
exigir(huella_claves(verificado)==huella_claves(detalle_cv), 'Claves alteradas en exportación')
exigir(verificado[CLAVE].equals(verificado[CLAVE].sort_values(CLAVE,kind='stable').reset_index(drop=True)),
       'CSV no conserva el orden declarado')
exigir(np.array_equal(verificado.fold.to_numpy(),detalle_cv.fold.to_numpy()), 'Fold alterado al serializar')
reconciliar(verificado)
del verificado
particiones_folds.to_csv(SALIDA/'particiones_folds.csv.gz',index=False,compression='gzip')
tablas = {'metricas_globales.csv':metricas_globales,'metricas_horizonte.csv':metricas_horizonte,
          'metricas_fold.csv':metricas_fold,'metricas_fold_horizonte.csv':metricas_fold_horizonte,
          'metricas_diarias.csv':metricas_diarias,'metricas_estacion.csv':metricas_estacion,
          'metricas_filtros.csv':metricas_filtros,'estabilidad_temporal.csv':estabilidad,
          'resumen_folds.csv':df_resumen_cv,'cobertura_lags.csv':cobertura_lags,
          'ranking_macro_horizontes.csv':ranking,'auditoria_ajustes.csv':pd.DataFrame(auditoria_ajustes),
          'coeficientes_ridge.csv':pd.DataFrame(coeficientes_ridge)}
for nombre,tabla in tablas.items():tabla.to_csv(SALIDA/nombre,index=False,encoding='utf-8-sig')
(SALIDA/'parametros_baselines_folds.json').write_text(json.dumps(parametros_baselines,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
(SALIDA/'variables_modelo_lags.json').write_text(json.dumps(PREDICTORES,indent=2),encoding='utf-8')
descripciones = {
    'station_id':'Identificador textual de estación','target_time_utc':'Fecha/hora objetivo UTC',
    'horizon_target_hours':'Horizonte nominal de 12 o 24 horas','state':'Estado en datos fuente',
    'latitude':'Latitud en grados','longitude':'Longitud en grados','fecha':'Fecha objetivo UTC',
    'temperature_forecast_c':'Pronóstico original, °C','temperature_observed_c':'Observación, °C',
    'momento_emision_utc':'Máximo entre generación y captura del pronóstico',
    'lag_error_disponible_c':'Último error disponible válido, °C','lag_error_edad_h':'Edad del objetivo del antecedente, horas',
    'lag_error_faltante_flag':'1 si falta antecedente válido',
    'lag_objetivo_utc':'Objetivo del antecedente utilizado','lag_disponible_utc':'Disponibilidad estimada del antecedente',
    'fold':'Ventana temporal 1 a 4','conjunto':'cv_temporal; no mezclar con holdout/test',
    'error_real_c':'Observado menos pronóstico original, °C','b3_respaldo_global_flag':'1 si estación ausente del train del fold',
    'codigo':'Identificador corto del modelo','modelo':'Nombre legible','registros':'Cantidad de predicciones por modelo',
    'suma_error_absoluto_c':'Numerador aditivo del MAE','suma_error_cuadrado_c2':'Numerador aditivo del MSE',
    'suma_residual_c':'Numerador aditivo del sesgo','MAE_c':'Error absoluto medio, °C',
    'RMSE_c':'Raíz de media de cuadrados, °C','sesgo_c':'Media observado menos corregido, °C',
    'MAE_original_c':'MAE B0 sobre las mismas filas','MAE_B3_c':'MAE B3 sobre las mismas filas',
    'mejora_MAE_pct':'100*(1-MAE/MAE_B0); vacío si MAE_B0=0',
    'ventaja_vs_B3_c':'MAE_B3 menos MAE del modelo; positivo favorece modelo'}
for codigo in MODELOS:
    descripciones[f'correccion_{codigo}_c'] = f'Error predicho por {codigo} ajustado en train del fold, °C'
    descripciones[f'temperatura_{codigo}_c'] = f'Pronóstico más corrección {codigo}, °C'
    descripciones[f'error_residual_{codigo}_c'] = f'Observado menos temperatura corregida {codigo}, °C'
campos = set(detalle_cv.columns)|set(metricas_filtros.columns)
exigir(campos<=set(descripciones), 'Campo del dashboard sin descripción')
pd.DataFrame([{'columna':c,'descripcion':descripciones[c]} for c in sorted(campos)]).to_csv(
    SALIDA/'diccionario_dashboard.csv',index=False,encoding='utf-8-sig')
print('Archivos exportados y reconciliados. Ejecutar la última celda para completar el manifiesto.')


## Celda 8 — Markdown · Conclusiones calculadas y alcance

La conclusión siguiente informa el modelo con menor MAE medio entre horizontes, los resultados globales y el número de folds con mejora. No presupone el signo del cambio ni un ganador. El ranking es descriptivo: no dispara reentrenamiento ni reemplaza el artefacto final del 04.

La CV mide configuraciones fijas bajo disponibilidad estimada y antecedentes secuenciales. No demuestra rendimiento prospectivo, independencia de todas las decisiones previas ni generalización a otra estación del año. Mejorar MAE/RMSE no implica eliminar sesgo. Un modelo complejo debe justificar su aporte frente a B3, no solo frente al original.

Se publica `manifest.json` únicamente después de reconciliar salidas y comprobar que las fuentes no cambiaron. Contiene hashes SHA-256 de todos los archivos exportados salvo el propio manifiesto, evitando una referencia circular. Una ejecución sin ese manifiesto debe excluirse del dashboard.


In [ ]:
# Celda 8 — Code | Conclusiones actuales y publicación final
minimo = ranking.MAE_macro_horizontes_c.min()
ganadores = ranking.loc[np.isclose(ranking.MAE_macro_horizontes_c,minimo,atol=1e-12,rtol=0),'modelo'].tolist()
lineas = [f'CV temporal retrospectiva: {len(detalle_cv):,} predicciones únicas en cuatro folds.',
          'Menor MAE medio entre horizontes (empates incluidos): '+', '.join(ganadores)+f' ({minimo:.4f} °C).']
b0 = metricas_globales.set_index('codigo').loc['b0']
for fila in metricas_globales.itertuples():
    mejora = f'{fila.mejora_MAE_pct:+.2f} %' if np.isfinite(fila.mejora_MAE_pct) else 'no definida (MAE B0=0)'
    lineas.append(f'{fila.modelo}: MAE {fila.MAE_c:.4f} °C; RMSE {fila.RMSE_c:.4f} °C; '
                  f'sesgo {fila.sesgo_c:+.4f} °C; reducción MAE {mejora}; ventaja frente a B3 {fila.ventaja_vs_B3_c:+.4f} °C.')
    if fila.MAE_c<b0.MAE_c and abs(fila.sesgo_c)>abs(b0.sesgo_c):
        lineas.append(f'{fila.modelo} mejora MAE pero aumenta el sesgo absoluto frente al original.')
for codigo in ['ridge_lags','hgb_lags','b3']:
    t = estabilidad.loc[estabilidad.codigo.eq(codigo)]
    lineas.append(MODELOS[codigo]+': '+ '; '.join(f'{r.horizon_target_hours} h: mejora en {r.folds_mejora_B0}/{r.folds_evaluados} folds' for r in t.itertuples()))
lineas += ['Todos los ajustes se hicieron dentro del train purgado de cada fold.',
           'No se utilizó noviembre como test ni se modificó el modelo final del 04.',
           'Configuraciones fijas exploradas previamente: no es CV anidada ni prueba prospectiva inédita.']
texto = chr(10).join(lineas)
print(texto)
(SALIDA/'conclusiones.txt').write_text(texto+chr(10),encoding='utf-8')
for ruta,clave in [(carpeta_04/'manifest.json','manifest_sha256'),(ruta_features,'features_sha256'),(ruta_variables,'variables_sha256')]:
    exigir(sha256_archivo(ruta)==fuente[clave], 'Fuente 04 modificada durante la ejecución')
manifest = {'estado':'completo','version':VERSION,'id_ejecucion':id_ejecucion,
    'inicio_utc':fecha_inicio_ejecucion.isoformat(),'fin_utc':datetime.now(timezone.utc).isoformat(),'fuente':fuente,
    'versiones':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__},
    'protocolo':{'tipo':'cv_temporal_retrospectiva_configuracion_fija','ventanas':VENTANAS,
       'corte_cv_exclusivo':str(CORTE_VAL),'purga':'error_disponible_utc < primera_emision_validacion',
       'lags':'secuencial_eventos_disponibles','margen_observacion_h':MARGEN_H,'edad_maxima_lag_h':EDAD_MAX_H,
       'coincidencias_exactas':False,'fuerza_B3':FUERZA_B3,'seed':SEED,
       'ajuste_preprocesador':'independiente por modelo y fold; equivalente al 04 actual',
       'precision':'float64; sin conversion deliberada a float32',
       'clave':CLAVE,'orden_exportacion':CLAVE,'ranking':'media MAE de 12 y 24 horas; descriptivo'},
    'conteos':{'historico':len(df_ml),'candidato_cv':len(df_cv),'predicciones_unicas_cv':len(detalle_cv),
               'folds':len(folds_temporales),'modelos_por_fold':len(MODELOS),'test_evaluado':0},
    'fechas_predicciones':{'desde':str(detalle_cv.target_time_utc.min()),'hasta':str(detalle_cv.target_time_utc.max())},
    'folds':resumen_folds,'parametros_estimadores':parametros_estimadores,
    'huellas_particiones_04_05':huellas_04_05,'huella_claves_cv_sha256':huella_claves(detalle_cv),
    'convencion_huella':'Igual a 05: CSV UTF-8, LF, sin índice; orden CLAVE; fecha UTC con microsegundos y Z',
    'verificaciones':{'features_reconciliadas_04':True,'lags_reconciliados_04':True,
                      'metricas_reconciliadas':True,'serializacion_reconciliada':True,'atol':1e-9,'rtol':0},
    'archivos':[{'nombre':p.name,'bytes':p.stat().st_size,'sha256':sha256_archivo(p)}
                for p in sorted(SALIDA.iterdir()) if p.is_file() and p.name not in ['manifest.json','manifest.tmp']]}
temporal = SALIDA/'manifest.tmp'
temporal.write_text(json.dumps(manifest,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
temporal.replace(SALIDA/'manifest.json')
print('Ejecución completa:',SALIDA)
